# Capital Allocation II

The ideas are in the slides. Here you apply the same formula to hedged portfolios, then put your own numbers into it.

**The plan**

1. 🎯 Alpha bets on five factors: hedge, size, combine with the market
2. All 29 strategies: the rule against the whole matrix
3. 🎯 Your views: a taste of Black–Litterman. Then back to the slides
4. 🎯 Challenge (homework): four trading ideas

📑 **Slides for this class:** [Capital Allocation II (PDF)](https://amoreira2.github.io/UG54/slides/capital_allocation_2.pdf)

<iframe src="../../slides/capital_allocation_2.pdf" width="100%" height="520" style="border: 1px solid #ddd;"></iframe>

In [ ]:
#@title Setup — run this first
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.figsize'] = [11, 4]
import warnings; warnings.filterwarnings('ignore')

BASE = "https://raw.githubusercontent.com/amoreira2/UG54/refs/heads/main/assets/data"

ff  = pd.read_csv(f"{BASE}/ff_monthly.csv", index_col=0, parse_dates=True)   # Ken French, monthly
L   = pd.read_parquet(f"{BASE}/longshort_29.parquet")                        # our 29 long-shorts, 1980-2000
ind = pd.read_parquet(f"{BASE}/ind49_monthly_1926.parquet")                  # 49 industries, for the challenge
fac = pd.read_parquet(f"{BASE}/factors_monthly_1926.parquet")                # Mkt-RF and RF, from 1926

F   = ff[['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'UMD']].dropna()             # the six factors, excess returns
mkt = F['Mkt-RF']

sharpe = lambda x: x.mean() / x.std() * np.sqrt(12)

print(f"six factors     {F.index[0]:%Y-%m} to {F.index[-1]:%Y-%m}, {len(F)} months")
print(f"29 long-shorts  {L.index[0]:%Y-%m} to {L.index[-1]:%Y-%m}, {len(L)} months")

---

## 1. 🎯 Alpha bets on five factors

Treat the market as something you already own, and the other five factors as your strategies. The table below does the two
steps from the slides. Read it before you run it.

In [ ]:
rows, H = {}, {}
for c in ['SMB', 'HML', 'RMW', 'CMA', 'UMD']:
    r = sm.OLS(F[c], sm.add_constant(mkt)).fit()
    H[c] = F[c] - r.params['Mkt-RF'] * mkt                        # 1. hedge: take the market out
    rows[c] = {'alpha': r.params['const'] * 12, 'beta': r.params['Mkt-RF'], 'sd_eps': r.resid.std() * np.sqrt(12)}
H = pd.DataFrame(H)
A = pd.DataFrame(rows).T

A['appraisal'] = A.alpha / A.sd_eps
A['w']         = A.alpha / A.sd_eps ** 2                          # 2. size: alpha over residual variance
A['vol_given'] = A.w * A.sd_eps

print(A.round(3).to_string())

- Compare `vol_given` and `appraisal`.
- `CMA` and `RMW` have the same appraisal ratio. Which gets the bigger weight, and why?
- Why is `SMB`'s weight negative?

**Now your turn.** Use `H` and `A.w`:

- build the book: one return per month, the hedged factors times their weights. Call it `book`
- find its Sharpe ratio, and the Sharpe ratio of the best mix of `book` and the market
- compare that with $\sqrt{SR_{m}^{2} + SR_{book}^{2}}$

In [ ]:
# === YOUR TURN ===
MY_PROMPT = """
                                    ← write your prompt here
"""

# ---- paste the AI's code below ----

book = ____

In [ ]:
#@title 🔒 Check
# the reference book: each hedged factor times its weight, added up month by month
book_ref = (H * A.w).sum(axis=1)

# put the market and the book side by side: two assets, so the formula works on a 2 x 2 problem
j = pd.concat([mkt.rename('market'), book_ref.rename('book')], axis=1)

# the tangency weights, in three steps: W = (covariance matrix)^-1 x (mean returns)
cov_j     = j.cov()                    # 1. the covariance matrix of the two, 2 x 2
cov_j_inv = np.linalg.inv(cov_j)       # 2. its inverse
w_mix     = cov_j_inv @ j.mean()       # 3. the inverse times the vector of mean returns
# monthly numbers are fine: per year, both would be 12 times bigger, and that rescales the weights
# without changing the mix, so the Sharpe ratio is the same

print(f"market                 Sharpe {sharpe(j.market):.2f}")
print(f"book                   Sharpe {sharpe(j.book):.2f}    correlation with the market {j.corr().iloc[0, 1]:+.3f}")
print(f"the two together       Sharpe {sharpe(j @ w_mix):.2f}")       # j @ w_mix: the mix, one return per month
print(f"sqrt(0.57^2 + 0.97^2)         {np.sqrt(sharpe(j.market) ** 2 + sharpe(j.book) ** 2):.2f}\n")

# the five hedged factors, this time using their whole covariance matrix: the same three steps
cov_H = H.cov()                                 # 1. covariance matrix, 5 x 5
w_H   = np.linalg.inv(cov_H) @ H.mean()         # 2. and 3. invert it, times the mean returns

print(f"if the five were uncorrelated: {np.sqrt((A.appraisal ** 2).sum()):.2f}")
print(f"using the whole matrix:        {sharpe(H @ w_H):.2f}\n")
print(H.corr().round(2).to_string())

try:
    print(f"\nyours: Sharpe {sharpe(book):.2f}   correlation with the reference {book.corr(book_ref):.3f}")
except Exception as e:
    print("\nyours: not built yet:", type(e).__name__)

- Does the formula hold for the book and the market? Why exactly, here?
- Adding the market to the book moved the Sharpe ratio from 0.97 to 1.12. Adding the book to the market moved it
  from 0.57 to 1.12. Why so different?
- The book falls short of what five uncorrelated strategies would give. Which pair is responsible? What angle is that?

---

## 2. All 29 strategies: the rule against the whole matrix

The same two steps on our 29 long-shorts, 1980–2000. Then the same formula without assuming the residuals are uncorrelated.

In [ ]:
mk = ff.loc[L.index, 'Mkt-RF']
fits = {c: sm.OLS(L[c], sm.add_constant(mk)).fit() for c in L}
H29  = pd.DataFrame({c: L[c] - fits[c].params.iloc[1] * mk for c in L})     # hedged on the market, full sample
a29  = pd.Series({c: fits[c].params.iloc[0] * 12 for c in L})               # alpha, a year
s29  = pd.Series({c: fits[c].resid.std() * np.sqrt(12) for c in L})         # residual volatility, a year

gross1 = lambda w: w / np.abs(w).sum()                                      # scale so |w| adds to one

# the rule: assume the residuals are uncorrelated, so each weight is alpha over residual variance
w_rule = gross1(a29 / s29 ** 2)

# the whole matrix: the same formula with all 29 x 29 covariances, in three steps
cov29     = H29.cov() * 12                                                  # 1. covariance matrix, a year
cov29_inv = np.linalg.inv(cov29)                                            # 2. its inverse
w_full    = gross1(pd.Series(cov29_inv @ a29, index=L.columns))             # 3. times the alphas, then scale

print(f"the rule:         Sharpe {sharpe(H29 @ w_rule):.2f}")
print(f"the whole matrix: Sharpe {sharpe(H29 @ w_full):.2f}")
print(f"if uncorrelated:  sqrt(sum AR^2) = {np.sqrt(((a29 / s29) ** 2).sum()):.2f}   (these two matching is a coincidence)\n")
print(pd.DataFrame({'alpha': a29, 'sd_eps': s29, 'the rule': w_rule, 'whole matrix': w_full})
        .sort_values('the rule', ascending=False).head(6).round(3).to_string())
print('...')
print(pd.DataFrame({'the rule': w_rule, 'whole matrix': w_full}).sort_values('whole matrix').head(4).round(3).to_string())

- `DivSeason` tops the list. Is it because of its alpha? Look at the other column.
- The whole matrix has a higher Sharpe ratio. Would you run it? Look at what it shorts, and at what those strategies are.
- These are in-sample numbers on signals that are on the list because they worked. What would you change before you
  trusted any of them? That is *Capital Allocation III*.

---

## 3. 🎯 Your views: a taste of Black–Litterman

The formula has no opinion about where $\alpha$ comes from. So start from the prior that there is no alpha at all, and add only
what you are willing to write down.

`bl` takes the prior, the covariance, your views and how unsure you are. Each row of `P` says which portfolio a view is about,
`Q` is the number, and `omega` is the standard error of each view.

In [ ]:
def bl(Sigma, P, Q, omega, tau=0.05):
    """Black-Litterman with a prior of zero alpha. Returns the alpha you act on."""
    P  = np.atleast_2d(P)                          # each row: the portfolio a view is about
    Q  = np.atleast_1d(Q)                          # each entry: your number for that portfolio
    Om = np.diag(np.atleast_1d(omega) ** 2)        # how unsure you are about each view, as a variance

    prior_cov = tau * Sigma                        # how unsure the prior of zero alpha is
    view_cov  = P @ prior_cov @ P.T + Om           # total uncertainty about each view: the prior's plus yours
    # move the alpha from zero toward your view, more the surer you are than the prior
    return prior_cov @ P.T @ np.linalg.inv(view_cov) @ Q

Sig5 = H.cov().values * 12        # the five hedged factors: covariance matrix estimated from data, a year

**Your turn.** Write one view on the five hedged factors, in the cell below, by editing the three lines at the top. For example:
*value beats the market by 4% a year over the next five years, give or take 3%*. Then see what the formula does with it.

Try it with a smaller `omega`, and with a view on a *spread*: `P = [0, 0, 0, 1, -1]` says RMW beats CMA.

In [ ]:
# === EDIT ===
P     = [0, 1, 0, 0, 0]     # which portfolio: order is SMB, HML, RMW, CMA, UMD
Q     = 0.04                # your number, a year
omega = 0.03                # how wrong you could be, a year

alpha_act = pd.Series(bl(Sig5, P, Q, omega), index=H.columns)

# the formula, gamma = 1: (covariance matrix)^-1 x alpha. Invert once, use it for both alphas
Sig5_inv = np.linalg.inv(Sig5)
w_view = pd.Series(Sig5_inv @ alpha_act, index=H.columns)     # weights from the alpha you act on
w_data = pd.Series(Sig5_inv @ A.alpha, index=H.columns)       # weights from the alpha in the data

table = pd.DataFrame({'alpha from data': A.alpha, 'alpha you act on': alpha_act,
                      'weights from data': w_data, 'weights from your view': w_view})
print((table.round(3) + 0.0).to_string())      # + 0.0 turns a rounding -0.000 into 0.000

- You wrote a view on one portfolio. The other *alphas* moved. Did the other *weights*? Compare the two columns, and say why.
- What happened to the size of your number? Make `omega` smaller and larger: what moves, the mix or the size?
- Which column would you rather run: 46 years of data, or one sentence? What would you want to do instead?

Bring your answer to that last question back to the slides.

---

## ⏸ Back to the slides

The challenge below is homework. It uses the second half of the slides.

---

## 🎯 Challenge: four trading ideas

*Homework — due before the next class. Have the discussion from the slides first.*

Cook up **four** trading ideas using the 49 industries in `ind`, and run them through the whole process. Each idea is a long-short
between groups of industries, hedged on the market. The order matters: **you write the number before you look at the data.**

### Q1 — The ideas, before any data

Fill in `IDEAS` below. For each one: the idea in a sentence, why it is not already in the price, the horizon, how you got your alpha
number, and how wrong you could be. Then fill in `BOOK` with the same four. `alpha` and `omega` are per year: if your idea has a shorter
horizon, say so in `IDEAS`, and say how you turned it into a yearly number.

Example, not one of your four: *"Higher-for-longer interest rates: long `Banks`, short `Util`. Consensus expects cuts. Horizon one year. A 1% rate
surprise has moved this pair about 3% (check it). I expect that surprise and I am right 60% of the time, so $(0.6-0.4)\times 3\% = 0.6\%$: `alpha` 0.006.
I could easily be wrong by 2%: `omega` 0.02."*

In [ ]:
IDEAS = """
Idea 1:
Idea 2:
Idea 3:
Idea 4:
"""

BOOK = {
    'idea 1': {'long': ['____'], 'short': ['____'], 'alpha': ____, 'omega': ____},
    'idea 2': {'long': ['____'], 'short': ['____'], 'alpha': ____, 'omega': ____},
    'idea 3': {'long': ['____'], 'short': ['____'], 'alpha': ____, 'omega': ____},
    'idea 4': {'long': ['____'], 'short': ['____'], 'alpha': ____, 'omega': ____},
}
n_ideas = len(BOOK)

### Q2 — Estimate the risk

Build each idea's monthly return: the average of its long industries minus the average of its short industries. Over **2016–2026**,
regress each on the market, and hedge it. Report each idea's $\beta$ and $\sigma_\varepsilon$, and the correlation matrix of the hedged ideas.

Call the annualized covariance matrix of the hedged ideas `S_eps`.

> Are these four ideas? If two are correlated above 0.7, they are one idea. Say what you would do about it, then do it.

In [ ]:
# Your work here


S_eps = ____

### Q3 — Calibrate the alpha

For each idea, three numbers side by side:

1. **Your number**, from Q1
2. **The record**: its alpha and t-statistic over the longest history you have. Does your number survive it? *Do not go back and change Q1:* say
   what you would change, and why
3. **The alpha you act on**: `bl(S_eps, np.eye(4), alphas, omegas)`, with the `bl` from class

Also the check from class: $\alpha/\sigma_\varepsilon$ at your horizon is the IC you are claiming. Which one is the most immodest?

In [ ]:
# Your work here


alpha_act = ____

### Q4 — Size the book

Use the formula on the alpha you act on and on `S_eps`, and scale the book to 10% volatility. Report the weights, the Sharpe ratio your inputs
promise, $\sqrt{\alpha' \Sigma_\varepsilon^{-1} \alpha}$, and the share of the gross exposure in your biggest position.

Now cut the alpha of your favourite idea in half and re-size. How much do the weights move?

> **📌 Required variable names:**
> ```python
> book_vol       = ____   # the book's volatility from your weights and S_eps: should be 0.10
> ex_ante_sharpe = ____   # the Sharpe ratio your inputs promise
> top_share      = ____   # largest |weight| / sum of |weights|
> ```

In [ ]:
# Your work here


book_vol       = ____
ex_ante_sharpe = ____
top_share      = ____

print(book_vol, ex_ante_sharpe, top_share)

### Q5 — What the past would have done

Run your book over 2016–2026, with the betas and weights from above, and report its Sharpe ratio. Compare it with `ex_ante_sharpe`.

Say what this number can and cannot tell you. You chose these ideas in 2026, knowing what happened.

> **📌 Required variable names:**
> ```python
> realized_sharpe = ____   # Sharpe ratio of the book over 2016-2026
> ```

In [ ]:
# Your work here


realized_sharpe = ____

print(realized_sharpe)

### Q6 — The memo

> **📝 Your task — maximum ten sentences.**
>
> You present this book to your PM. Defend your biggest position: what you believe, why the market does not, by when you will know, and what would
> make you close it. Say where each alpha came from: a record, a signal or a hunch. Say which number in the whole process you trust least. Finish with
> what you would write down today so that a year from now you can tell whether you had any skill.

In [ ]:
MEMO = """
Write your memo here. Don't delete the surrounding triple quotes.
"""
print(MEMO)

---

## 📤 Submission

In [ ]:
# === 📤 SUBMISSION CELL — Run this last ===
import json, base64, hashlib, datetime as dt

required = ["n_ideas", "book_vol", "ex_ante_sharpe", "top_share", "realized_sharpe", "MEMO"]
missing = [v for v in required if v not in globals()]
if missing:
    raise NameError(f"\n❌ Missing before submission: {missing}")

payload = {
    "assignment": "L12b_CapitalAllocationII_v2_AI",
    "ts": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z"),
    "answers": {k: float(eval(k)) for k in required if k != "MEMO"},
    "memo": (IDEAS + "\n\n" + MEMO).strip(),
}
blob = json.dumps(payload, sort_keys=True)
checksum = hashlib.sha256(blob.encode()).hexdigest()[:8]
token = f"UG54::{checksum}::{base64.b64encode(blob.encode()).decode()}"

print("=" * 72)
print("📋  COPY THE LINE BELOW AND PASTE INTO THE SUBMISSION FORM")
print("=" * 72)
print(token)
print("=" * 72)
print("Submission form: https://forms.gle/yazZ8bbatL87jdJi7")